<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 粒子群最佳化（PSO）：原課程 MATLAB 動畫程式的 Python 版

對照網頁：[粒子群最佳化](https://yoshiagent.github.io/IntControl/PSO.html)。原講義附檔 `PSO animations.zip` 有四支程式（Shu-Hao Lu，2004）：

| 檔案 | 函數 | 畫法 |
|---|---|---|
| PSO_02_ex01_animation01.m | 範例一：−20 sin r / r，r = 0.1 + √((x−4)² + (y−4)²)，−10 ≤ x, y ≤ 10 | 等高線 |
| PSO_02_ex01_animation02.m | 同上 | 3D 曲面 |
| PSO_02_ex02_animation01.m | 範例二：x sin(4x) + 1.1 y sin(2y)，0 ≤ x, y ≤ 10 | 等高線 |
| PSO_02_ex02_animation02.m | 同上 | 3D 曲面 |

四支程式的搜尋演算法完全相同，只有函數與畫圖方式不同，所以下面把演算法寫成一個函式，再分別畫等高線與 3D 動畫（按 ▶ 播放，或拖拉下方的滑桿看每一代）。
只用 Colab 內建的 numpy 與 plotly。MATLAB 與 numpy 的亂數序列不同，每次執行的過程不會和原講義一模一樣。

In [1]:
import numpy as np
import plotly.graph_objects as go

## 1. 原程式（範例一、等高線版）

這是 PSO_02_ex01_animation01.m 去掉畫圖指令（`line`、`set`、`pause`…）後的計算部分：

```matlab
PopuSize = 20;      % population size
var_n = 2;          % total number of design variables
gen_no = 40;        % set the maximum iteration number
range=[-10 -10; ...
        10  10];    % side bounds for each design variable
Vmax=(range(2,:)-range(1,:))*0.5;   % set the max velocity as half the range
c1 = 2;             % acceleration constant c1
c2 = 2;             % acceleration constant c2
w = 0.9:-(0.9-0.4)/gen_no:0.4;  % inertial weight for updating velocity
...
x = range(1,1) + (range(2,1)-range(1,1)) * rand(PopuSize,1);
y = range(1,2) + (range(2,2)-range(1,2)) * rand(PopuSize,1);
Vx = Vmax(1) * rand(PopuSize,1);
Vy = Vmax(2) * rand(PopuSize,1);
V = [Vx, Vy];
fit_gbest = 0;                  % set the initial value for fit_gbest
fit_pbest = zeros(PopuSize,1);  % set the initial value for fit_pbest of each particle

for current_gen=1:gen_no		% for each generation => BEGIN
for j=1:PopuSize                % for each particle => BEGIN
fxy = -20*sin(0.1+((x(j)-4).^2+(y(j)-4).^2).^0.5)./(0.1+((x(j)-4).^2+(y(j)-4).^2).^0.5);
fitness(j) = -fxy;              % transform the problem from minimization to maximization.
if fitness(j) > fit_gbest
    fit_gbest = fitness(j);
    gbest = [x(j), y(j)];
end
if fitness(j) > fit_pbest(j)
    fit_pbest(j) = fitness(j);
    pbest(j,:) = [x(j), y(j)];
end
x_original(j) = x(j);
y_original(j) = y(j);
V(j,:) = w(current_gen)*V(j,:) + c1*rand*(gbest-[x(j),y(j)]) + c2*rand*(pbest(j,:)-[x(j),y(j)]);
for i=1:var_n
    if V(j,i) > Vmax(i)
        V(j,i) = Vmax(i);
    end
end
x(j) = x(j) + V(j,1);
y(j) = y(j) + V(j,2);
if x(j) < range(1,1)
    x(j) = range(1,1);
elseif x(j) > range(2,1);
    x(j) = range(2,1);
end
if y(j) < range(1,2)
    y(j) = range(1,2);
elseif y(j) > range(2,2);
    y(j) = range(2,2);
end
end                             % for each particle => END
[fit,ind] = sort(fitness);      % sort for fitness
clear fit;
bestxy(current_gen,:) = [x_original(ind(PopuSize)), y_original(ind(PopuSize))];
bestfit(current_gen,1) = fitness(ind(PopuSize));
end		                        % for each generation => END
```

幾個細節：

* 粒子是**一個一個**更新的（非同步）：前面的粒子一找到更好的位置，gbest 就立刻更新，後面的粒子馬上用新的 gbest。
* `c1*rand*(gbest-[x(j),y(j)])`：同一個亂數乘在 x、y 兩個方向上。
* 慣性權重 w 從 0.9 線性遞減到 0.4（`w` 有 gen_no + 1 = 41 個值，只用到前 40 個）。
* 這裡的 c₁ 配 gBest、c₂ 配 pBest，和多數教科書相反；兩者都取 2，沒有差別。

In [2]:
f_ex01 = lambda x, y: -20 * np.sin(0.1 + np.sqrt((x - 4) ** 2 + (y - 4) ** 2)) / (0.1 + np.sqrt((x - 4) ** 2 + (y - 4) ** 2))
f_ex02 = lambda x, y: x * np.sin(4 * x) + 1.1 * y * np.sin(2 * y)

def pso_original(f, range_, seed, PopuSize=20, gen_no=40, c1=2.0, c2=2.0, clamp_both=False, pbest_from_start=False):
    """照原程式逐行改寫。clamp_both、pbest_from_start 是第 4 節的兩個修正（預設 False＝和原程式相同）。
    回傳每一代開始時的位置（41 張快照，給動畫用）、當時的 gbest、每代的 bestfit 與 bestxy"""
    rng = np.random.default_rng(seed)
    lo, hi = np.array(range_[0], float), np.array(range_[1], float)
    Vmax = (hi - lo) * 0.5
    w = np.linspace(0.9, 0.4, gen_no + 1)                         # 0.9:-(0.9-0.4)/gen_no:0.4
    x = lo[0] + (hi[0] - lo[0]) * rng.random(PopuSize)
    y = lo[1] + (hi[1] - lo[1]) * rng.random(PopuSize)
    V = np.c_[Vmax[0] * rng.random(PopuSize), Vmax[1] * rng.random(PopuSize)]
    fit_gbest, gbest = 0.0, np.zeros(2)
    fit_pbest, pbest = np.zeros(PopuSize), np.zeros((PopuSize, 2))
    if pbest_from_start:                                          # 修正：pBest 從各粒子的起點開始
        fit_pbest, pbest = np.full(PopuSize, -np.inf), np.c_[x, y]
        fit_gbest = -np.inf
    fitness, xo, yo = np.zeros(PopuSize), np.zeros(PopuSize), np.zeros(PopuSize)
    frames, gframes, bestfit, bestxy = [np.c_[x, y].copy()], [gbest.copy()], [], []
    for g in range(gen_no):
        for j in range(PopuSize):
            fitness[j] = -f(x[j], y[j])                           # 求最小值 → 轉成求最大值
            if fitness[j] > fit_gbest:
                fit_gbest, gbest = fitness[j], np.array([x[j], y[j]])
            if fitness[j] > fit_pbest[j]:
                fit_pbest[j], pbest[j] = fitness[j], [x[j], y[j]]
            xo[j], yo[j] = x[j], y[j]
            V[j] = w[g] * V[j] + c1 * rng.random() * (gbest - [x[j], y[j]]) + c2 * rng.random() * (pbest[j] - [x[j], y[j]])
            V[j] = np.clip(V[j], -Vmax, Vmax) if clamp_both else np.minimum(V[j], Vmax)   # 原程式只限制上限
            x[j] = min(max(x[j] + V[j, 0], lo[0]), hi[0])
            y[j] = min(max(y[j] + V[j, 1], lo[1]), hi[1])
        k = int(np.argsort(fitness, kind="stable")[-1])
        bestxy.append([xo[k], yo[k]]); bestfit.append(fitness[k])
        frames.append(np.c_[x, y].copy()); gframes.append(gbest.copy())
    return dict(frames=frames, gbest=gframes, bestfit=np.array(bestfit), bestxy=np.array(bestxy))

def report(r, name, x_ans, y_ans, bestfit_ans):
    """和原程式最後的 disp 相同"""
    k = int(np.argsort(r["bestfit"], kind="stable")[-1])
    x, y, b = r["bestxy"][k, 0], r["bestxy"][k, 1], r["bestfit"].max()
    print(f"Filename: {name}")
    print(f"   PopuSize: 20        gen_no: 40")
    print(f"   max(bestfit) = {b:.4f}      bestfit_error = {abs(bestfit_ans - b) / bestfit_ans * 100:.4g} (%)")
    print(f"   x = {x:.4f}       x_error = {abs(x_ans - x) / x_ans * 100:.4g} (%)")
    print(f"   y = {y:.4f}       y_error = {abs(y_ans - y) / y_ans * 100:.4g} (%)")

r1 = pso_original(f_ex01, [[-10, -10], [10, 10]], seed=1)
report(r1, "PSO_02_ex01_animation01", 4, 4, 19.9667)

Filename: PSO_02_ex01_animation01
   PopuSize: 20        gen_no: 40
   max(bestfit) = 19.9649      bestfit_error = 0.009004 (%)
   x = 3.9974       x_error = 0.06546 (%)
   y = 3.9997       y_error = 0.00834 (%)


### 等高線動畫（animation01）

原程式用 `contour` 畫函數，黑色六角形是真正的最小值 (4, 4)，藍點是粒子、紅色 × 是 gBEST。每畫一個粒子就 `pause(0.05)`；這裡改成每一代一張畫面。

In [3]:
def contour_animation(r, f, range_, step, opt, title):
    gx = np.arange(range_[0][0], range_[1][0] + step / 2, step)
    gy = np.arange(range_[0][1], range_[1][1] + step / 2, step)
    X, Y = np.meshgrid(gx, gy)
    base = [go.Contour(x=gx, y=gy, z=np.round(f(X, Y), 3), colorscale="Viridis", showscale=False, opacity=0.85),
            go.Scatter(x=[opt[0]], y=[opt[1]], mode="markers", marker=dict(symbol="hexagon", size=18, color="black"), name="optimum")]
    def pts(k):
        p, g = r["frames"][k], r["gbest"][k]
        return [go.Scatter(x=p[:, 0], y=p[:, 1], mode="markers+text", text=[str(i + 1) for i in range(len(p))], textposition="top right",
                           marker=dict(color="blue", size=9), textfont=dict(color="blue", size=10), name="particles"),
                go.Scatter(x=[g[0]], y=[g[1]], mode="markers+text", text=["gBEST"], textposition="top right",
                           marker=dict(symbol="x", size=16, color="red"), textfont=dict(color="red"), name="gBEST")]
    n = len(r["frames"])
    fig = go.Figure(data=base + pts(0), frames=[go.Frame(data=pts(k), traces=[2, 3], name=str(k)) for k in range(n)])
    fig.update_layout(
        title=title, xaxis=dict(range=[range_[0][0], range_[1][0]], title="X"), yaxis=dict(range=[range_[0][1], range_[1][1]], title="Y", scaleanchor="x"),
        width=620, height=620, showlegend=False,
        updatemenus=[dict(type="buttons", x=0, y=-0.08, xanchor="left", buttons=[
            dict(label="▶ play", method="animate", args=[None, dict(frame=dict(duration=300, redraw=False), fromcurrent=True)]),
            dict(label="❚❚ pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0))])])],
        sliders=[dict(active=0, x=0.2, len=0.8, y=-0.04, currentvalue=dict(prefix="generation "),
                      steps=[dict(label=str(k), method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=False))]) for k in range(n)])])
    return fig

contour_animation(r1, f_ex01, [[-10, -10], [10, 10]], 0.5, (4, 4), "P.S.O. Animation (example 1)").show()

### 3D 曲面動畫（animation02）

原程式用 `surfc(X,Y,-FXY)` 畫適應值（−f）的曲面，粒子畫在它的適應值高度上：

```matlab
surfc(X,Y,-FXY);
colormap winter
view(-20,20);
...
set(Hagent(j),'xdata',x(j),'ydata',y(j),'zdata',fitness(j),'visible','on');
```

In [4]:
def surface_animation(r, f, range_, step, opt, title, zmin):
    gx = np.arange(range_[0][0], range_[1][0] + step / 2, step)
    gy = np.arange(range_[0][1], range_[1][1] + step / 2, step)
    X, Y = np.meshgrid(gx, gy)
    def pts(k):
        p, g = r["frames"][k], r["gbest"][k]
        return [go.Scatter3d(x=p[:, 0], y=p[:, 1], z=-f(p[:, 0], p[:, 1]), mode="markers", marker=dict(color="blue", size=4), name="particles"),
                go.Scatter3d(x=[g[0]], y=[g[1]], z=[-f(g[0], g[1])], mode="markers", marker=dict(color="red", size=7, symbol="x"), name="gBEST")]
    base = [go.Surface(x=gx, y=gy, z=np.round(-f(X, Y), 3), colorscale="Blues", opacity=0.8, showscale=False)]
    n = len(r["frames"])
    fig = go.Figure(data=base + pts(0), frames=[go.Frame(data=pts(k), traces=[1, 2], name=str(k)) for k in range(n)])
    fig.update_layout(
        title=title, width=680, height=600, showlegend=False,
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Fitness", zaxis=dict(range=[zmin, None]),
                   camera=dict(eye=dict(x=1.2, y=-1.6, z=0.9))),
        updatemenus=[dict(type="buttons", x=0, y=0, xanchor="left", buttons=[
            dict(label="▶ play", method="animate", args=[None, dict(frame=dict(duration=300, redraw=True), fromcurrent=True)]),
            dict(label="❚❚ pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0))])])],
        sliders=[dict(active=0, x=0.2, len=0.8, y=0, currentvalue=dict(prefix="generation "),
                      steps=[dict(label=str(k), method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in range(n)])])
    return fig

surface_animation(r1, f_ex01, [[-10, -10], [10, 10]], 0.5, (4, 4), "P.S.O. Animation (example 1, surface)", -5).show()

## 2. 範例二（ex02）：x sin(4x) + 1.1 y sin(2y)

和範例一的差別只有這幾行：

```matlab
range=[  0   0; ...
        10  10];
fxy = x(j).*sin(4*x(j))+1.1*y(j).*sin(2*y(j));
    x_ans=9.0390;
    y_ans=8.6682;
    bestfit_ans=18.5547;
```
程式同樣以 fitness = −f 轉成求最大值，所以找的是 f 的**最小值** −18.5547（在 (9.039, 8.668)）。這個函數有很多個谷，比範例一難。

In [5]:
r2 = pso_original(f_ex02, [[0, 0], [10, 10]], seed=2)
report(r2, "PSO_02_ex02_animation01", 9.0390, 8.6682, 18.5547)
contour_animation(r2, f_ex02, [[0, 0], [10, 10]], 0.15, (9.0390, 8.6682), "P.S.O. Animation (example 2)").show()   # 原程式格點 0.1；這裡用 0.15 讓檔案小一點

Filename: PSO_02_ex02_animation01
   PopuSize: 20        gen_no: 40
   max(bestfit) = 18.5547      bestfit_error = 0.0002148 (%)
   x = 9.0397       x_error = 0.007815 (%)
   y = 8.6693       y_error = 0.01277 (%)


In [6]:
surface_animation(r2, f_ex02, [[0, 0], [10, 10]], 0.15, (9.0390, 8.6682), "P.S.O. Animation (example 2, surface)", -20).show()

### 每一代的最佳適應值

原程式最後畫 `plot(1:gen_no,bestfit,'o',1:gen_no,bestfit,'-')`。注意 `bestfit` 是「**這一代**粒子中最好的」，不是到目前為止的最好值，所以曲線會上下跳動：

In [7]:
fig = go.Figure()
fig.add_scatter(x=np.arange(1, 41), y=r1["bestfit"], mode="lines+markers", name="example 1 (answer 19.9667)")
fig.add_scatter(x=np.arange(1, 41), y=r2["bestfit"], mode="lines+markers", name="example 2 (answer 18.5547)")
fig.update_layout(title="Particle Swarm Optimization", xaxis_title="generation", yaxis_title="best fitness of each generation", height=360)
fig.show()

## 3. 原程式的兩個問題

1. **速度只限制上限**：`if V(j,i) > Vmax(i)` 沒有處理負方向，往左、往下飛的粒子速度可以遠超過 Vmax。應改為 `V(j,i) = max(-Vmax(i), min(Vmax(i), V(j,i)))`。
2. **pBest 的初值是 (0, 0)、適應值 0**：`fit_pbest = zeros(PopuSize,1)`，而 `pbest` 也預設為 0。若粒子一直待在適應值為負的地方（範例一 r 在 π～2π 的環、範例二 f > 0 的區域），pBest 就一直停在 (0, 0)，速度公式會把它往 (0, 0) 拉，而那裡根本不是它到過的位置。應讓 pBest 從粒子的起點開始（初值 −∞）。

下面各跑 200 次（不同的亂數種子），統計 40 代內找到最佳值（誤差 < 0.01）的比例，以及最負的速度：

In [8]:
def study(f, range_, ans, **kw):
    ok, vmin = 0, 0.0
    for s in range(200):
        r = pso_original(f, range_, seed=1000 + s, **kw)
        ok += abs(r["bestfit"].max() - ans) < 0.01
        v = np.diff(np.array(r["frames"]), axis=0)               # 每代的位移（含邊界夾住）
        vmin = min(vmin, v.min())
    return ok / 200, vmin

rows = []
for name, f, rg, ans in (("example 1", f_ex01, [[-10, -10], [10, 10]], 19.9667), ("example 2", f_ex02, [[0, 0], [10, 10]], 18.5547)):
    for label, kw in (("原程式", {}), ("速度雙向限制", dict(clamp_both=True)), ("pBest 從起點開始", dict(pbest_from_start=True)),
                      ("兩個都修正", dict(clamp_both=True, pbest_from_start=True))):
        rate, vmin = study(f, rg, ans, **kw)
        rows.append((name, label, rate, vmin))
        print(f"{name}  {label:10s}：成功率 {rate:5.1%}，最負的一代位移 {vmin:6.2f}（Vmax = {(rg[1][0] - rg[0][0]) / 2:g}）")

example 1  原程式       ：成功率 95.5%，最負的一代位移 -20.00（Vmax = 10）


example 1  速度雙向限制    ：成功率 97.0%，最負的一代位移 -10.00（Vmax = 10）


example 1  pBest 從起點開始：成功率 96.0%，最負的一代位移 -20.00（Vmax = 10）


example 1  兩個都修正     ：成功率 98.0%，最負的一代位移 -10.00（Vmax = 10）


example 2  原程式       ：成功率 81.0%，最負的一代位移 -10.00（Vmax = 5）


example 2  速度雙向限制    ：成功率 76.5%，最負的一代位移  -5.00（Vmax = 5）


example 2  pBest 從起點開始：成功率 83.5%，最負的一代位移 -10.00（Vmax = 5）


example 2  兩個都修正     ：成功率 83.0%，最負的一代位移  -5.00（Vmax = 5）


看上面的數字：

* 只限制上限時，粒子一代可以往負方向移動超過 Vmax（範例一甚至一步走完整個 −10～10 的範圍，被邊界擋住才停），修正後就不會超過 Vmax。
* 但在這兩個二維問題上，四種寫法的成功率差別都不大，有的修正甚至稍低。200 次的統計有誤差：成功率約 80% 時，兩種寫法的成功率之差的標準誤約 4 個百分點，
  差距要超過約 8 個百分點（2 倍標準誤）才算顯著；上面最大的差距是範例二的 7 個百分點，還不顯著。也就是說，這兩個問題剛好不太受影響。

修正的理由是讓程式符合演算法的原意（速度限制在 ±Vmax、pBest 是粒子自己到過的最好位置），不是因為在這兩個例子上一定會比較好。寫自己的 PSO 時，建議照修正版寫。

網站的 [PSO 頁](https://yoshiagent.github.io/IntControl/PSO.html#demo)有可以即時操作的互動版本；
[CMA-ES 一頁](https://yoshiagent.github.io/IntControl/OPT-CMAES.html)比較了 PSO、GA 與 CMA-ES 在 10 維問題上的表現。